# 05 — Eksploracyjna analiza ekspresji (EDA)

Eksploracyjny notebook **przed** modelami przeżycia. Cel: poznać strukturę danych ekspresji w kohorcie TCGA-LUAD przed budową baseline survival (notebook 06). Sprawdzić rzeczy, których w 80% prac DL się nie sprawdza, a recenzent o nie zapyta.

**Zakres:**

1. Budowa macierzy TPM (log2-transformed) — drugi wariant obok counts z notebooka 02
2. Sanity check rozkładu TPM
3. Top variable genes — materiał do feature selection
4. PCA — czy ekspresja niesie sygnał biologiczny (stage, event)
5. **Batch effect — TSS (Tissue Source Site)** — krytyczne dla obrony
6. Markery LUAD — biologiczny sanity check
7. Heatmapa koekspresji top variable genes
8. Wnioski + decyzje dla notebooka 06

**Pakiety:** poza tym co już mamy w repo, ten notebook wymaga `scikit-learn`, `matplotlib`, `seaborn`. Jeśli brak: `uv add scikit-learn matplotlib seaborn`.

**Co NIE wchodzi w zakres:** Kaplan-Meier, Cox, modelowanie. To jest tylko EDA. Modelowanie w notebooku 06.


## 1. Konfiguracja i wczytanie danych

In [ ]:
import sys
from pathlib import Path
from datetime import datetime, timezone

import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.transform import build_expression_matrix
from src.ingest import parse_sample_sheet, parse_clinical
from src.ingest.file_naming import STAR_FILE_PATTERNS

RAW_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim" / "star_counts"

TOP_N_GENES = 5000

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

print(f"Projekt: {PROJECT_ROOT}")
print(f"Polars: {pl.__version__} | NumPy: {np.__version__}")
print(f"Czas: {datetime.now(timezone.utc).isoformat()}")
print(f"TOP_N_GENES dla PCA i heatmapy: {TOP_N_GENES}")


In [ ]:
print("=== Wczytuję sample sheet i clinical ===")
sheet_path = next(RAW_DIR.glob("gdc_sample_sheet*.tsv"))
sheet = parse_sample_sheet(sheet_path)
print(f"Sample sheet: {sheet.height} próbek")

clinical_path = RAW_DIR / "clinical.tsv"
clinical = parse_clinical(clinical_path)
print(f"Clinical:     {clinical.height} pacjentów")


## 2. Budowa macierzy TPM (log2-transformed)

Notebook 02 zbudował macierz na `unstranded` (raw counts) — dobre do operacyjnej dokumentacji pipeline. Dla EDA i przyszłych modeli używamy **TPM z log2 transformacją** — standard w analizach bulk RNA-seq survival:

- TPM normalizuje na długość genu i głębokość sekwencjonowania (porównywalne między próbkami)
- log2(TPM+1) stabilizuje wariancję (kluczowe dla PCA i Cox)
- Każda jednostka log2 = 2-fold change w ekspresji (interpretowalność dla Cox HR)


In [ ]:
print("=== Buduję macierz TPM ===")
parquets = sorted(INTERIM_DIR.rglob("*.parquet"))
print(f"Parquetów: {len(parquets)}")
print()

matrix_tpm = build_expression_matrix(
    parquet_paths=parquets,
    sample_sheet=sheet,
    metric="tpm_unstranded",
    duplicate_strategy="deepest",
    biotype_filter="protein_coding",
)
print(f"Macierz TPM: {matrix_tpm.height} genów × {matrix_tpm.width - 1} próbek")


In [ ]:
print("=== Transformacja log2(TPM + 1) ===")
sample_cols = [c for c in matrix_tpm.columns if c != "gene_id"]

# Konwersja na numpy (geny x próbki) - łatwiejsze do log + PCA
gene_ids = matrix_tpm["gene_id"].to_list()
X = matrix_tpm.select(sample_cols).to_numpy()
print(f"Shape przed log: {X.shape}  | dtype: {X.dtype}")
print(f"  min={X.min():.4f}  max={X.max():.2f}  mean={X.mean():.4f}")

X_log = np.log2(X + 1)
print(f"Shape po log:    {X_log.shape}")
print(f"  min={X_log.min():.4f}  max={X_log.max():.2f}  mean={X_log.mean():.4f}")


## 3. Sanity check rozkładu TPM

Trzy rzeczy do sprawdzenia:

1. **Histogram TPM przed i po log** — czy log skutecznie stabilizuje rozkład
2. **Suma TPM per próbka** — z definicji TPM powinna być ≈ 10⁶. Jeśli różne wartości → problem z normalizacją albo próbki znacznie nietypowe
3. **Próbki-outliery** — czy są pacjenci z drastycznie innym rozkładem


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Przed log
axes[0].hist(X.flatten(), bins=80, color="#8b5a3c", edgecolor="white", alpha=0.85)
axes[0].set_xlim(0, 50)  # cap dla wizualizacji - długi ogon
axes[0].set_xlabel("TPM (cap: 50)")
axes[0].set_ylabel("Liczba (gen × próbka)")
axes[0].set_title(f"Rozkład TPM przed log\n(długi prawy ogon obcięty dla widoczności)")
axes[0].set_yscale("log")

# Po log
axes[1].hist(X_log.flatten(), bins=80, color="#5a8b3c", edgecolor="white", alpha=0.85)
axes[1].set_xlabel("log2(TPM + 1)")
axes[1].set_ylabel("Liczba (gen × próbka)")
axes[1].set_title("Rozkład log2(TPM + 1)\n(stabilizacja wariancji)")
axes[1].set_yscale("log")

plt.tight_layout()
plt.show()


In [ ]:
# Suma TPM per próbka - powinna być ~10^6 z definicji
sums = X.sum(axis=0)  # suma per kolumna (próbka)
print(f"=== Suma TPM per próbka ===")
print(f"  N próbek: {len(sums)}")
print(f"  Średnia: {sums.mean():,.0f}  (oczekiwane ~1,000,000)")
print(f"  Mediana: {np.median(sums):,.0f}")
print(f"  Min:     {sums.min():,.0f}")
print(f"  Max:     {sums.max():,.0f}")
print(f"  CV:      {sums.std() / sums.mean() * 100:.2f}%")
print()

# Outliery - próbki >5% od mediany
median_sum = np.median(sums)
deviation = np.abs(sums - median_sum) / median_sum
outliers_idx = np.where(deviation > 0.05)[0]
print(f"Próbki z sum TPM odchylonych >5% od mediany: {len(outliers_idx)}")
if len(outliers_idx) > 0 and len(outliers_idx) <= 20:
    for i in outliers_idx[:20]:
        print(f"  {sample_cols[i]:25} sum={sums[i]:,.0f}  ({(sums[i]/median_sum - 1)*100:+.1f}%)")


## 4. Top variable genes

Wariancja po log2 transformacji — geny które najmocniej różnicują pacjentów. To są kandydaci na predyktory survival.


In [ ]:
# Wariancja per gen (po log)
gene_vars = X_log.var(axis=1)
print(f"=== Wariancja log2(TPM+1) per gen ===")
print(f"  N genów: {len(gene_vars)}")
print(f"  Min:     {gene_vars.min():.4f}")
print(f"  Q25:     {np.percentile(gene_vars, 25):.4f}")
print(f"  Mediana: {np.median(gene_vars):.4f}")
print(f"  Q75:     {np.percentile(gene_vars, 75):.4f}")
print(f"  Max:     {gene_vars.max():.4f}")
print()

# Ile genów ma wariancję > różnych progów
for threshold in [0.1, 0.5, 1.0, 2.0]:
    n_above = (gene_vars > threshold).sum()
    print(f"  var > {threshold}: {n_above} genów ({n_above/len(gene_vars)*100:.1f}%)")


In [ ]:
# Indeksy top N (sortowanie po var, malejąco)
top_idx = np.argsort(gene_vars)[::-1][:TOP_N_GENES]
print(f"=== Top {TOP_N_GENES} najbardziej zmiennych genów ===")
print(f"  Variance range: {gene_vars[top_idx[-1]]:.3f} ... {gene_vars[top_idx[0]]:.3f}")
print()
print(f"Top 20 (po Ensembl ID):")
for rank, i in enumerate(top_idx[:20], 1):
    print(f"  {rank:2}. {gene_ids[i]:22}  var={gene_vars[i]:6.3f}  mean_log2tpm={X_log[i].mean():5.2f}")

# Wycinek dla PCA
X_log_top = X_log[top_idx, :]
print()
print(f"X_log_top shape: {X_log_top.shape}  (top genes × samples)")


## 5. PCA na top variable genes

Skalowanie (z-score per gen, żeby PCA nie był zdominowany przez geny o wysokiej średniej), PCA na 10 komponentów, scree plot + scatter PC1×PC2 kolorowany po kowariantach klinicznych.

**Co chcemy zobaczyć:**

- **Scree plot** — ile komponentów wystarcza (powinno spadać szybko, jeśli ekspresja niesie zbiorczy sygnał)
- **PC1×PC2 po stage** — czy zaawansowane stadia odróżniają się ekspresją
- **PC1×PC2 po event** — czy pacjenci ze zgonem mają inny profil ekspresji
- **PC1×PC2 po gender** — sprawdzenie czy nie ma niespodzianki (gender effect może być obecny w niektórych szlakach)


In [ ]:
# Skalowanie (z-score per gen) - standardowe pre-PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_log_top.T)  # transpozycja: próbki × geny
print(f"X_scaled shape: {X_scaled.shape}")

pca = PCA(n_components=10)
pcs = pca.fit_transform(X_scaled)
print(f"PCs shape: {pcs.shape}")
print()
print("=== Variance explained ===")
for i, var in enumerate(pca.explained_variance_ratio_, 1):
    print(f"  PC{i:2}: {var*100:5.2f}%  (cum: {pca.explained_variance_ratio_[:i].sum()*100:.1f}%)")


In [ ]:
# Scree plot
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].bar(range(1, 11), pca.explained_variance_ratio_ * 100, color="#5a8b3c", edgecolor="white")
axes[0].set_xlabel("Komponent")
axes[0].set_ylabel("Variance explained (%)")
axes[0].set_title("Scree plot")
axes[0].set_xticks(range(1, 11))

axes[1].plot(range(1, 11), np.cumsum(pca.explained_variance_ratio_) * 100, marker="o", color="#8b5a3c")
axes[1].set_xlabel("Komponent")
axes[1].set_ylabel("Cumulative variance (%)")
axes[1].set_title("Cumulative variance explained")
axes[1].set_xticks(range(1, 11))
axes[1].axhline(50, color="grey", linestyle="--", alpha=0.5, label="50%")
axes[1].axhline(80, color="grey", linestyle=":", alpha=0.5, label="80%")
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
# Mapping sample_id -> metadata
# UWAGA: sample_sheet zawiera 601 wierszy (po jednym per file_id), ale macierz
# ma 590 kolumn po deduplikacji aliquotów. Joinujemy + dedupujemy meta na sample_id.

sample_to_meta = sheet.select(["sample_id", "case_id", "tissue_type"]).unique(subset=["sample_id"])
clinical_short = clinical.select([
    "case_submitter_id", "vital_status", "time", "event",
    "age_at_index", "gender", "ajcc_pathologic_stage",
])

meta = (
    pl.DataFrame({"sample_id": sample_cols})
    .join(sample_to_meta, on="sample_id", how="left")
    .join(clinical_short, left_on="case_id", right_on="case_submitter_id", how="left")
)

# TSS - znaki 5-6 w barcode TCGA-XX-YYYY-... (np. "44" w TCGA-44-1234)
meta = meta.with_columns(
    pl.col("sample_id").str.slice(5, 2).alias("tss")
)

assert meta.height == len(sample_cols), f"meta height {meta.height} != sample_cols {len(sample_cols)}"
print(f"Meta dla {meta.height} próbek (zgodne z macierzą {len(sample_cols)})")
print()

# === Wszystkie numpy arrays w jednym miejscu - dostępne dla pozostałych komórek ===
stage_arr = meta["ajcc_pathologic_stage"].fill_null("Unknown").to_numpy()
event_arr = meta["event"].fill_null(False).to_numpy()
gender_arr = meta["gender"].fill_null("unknown").to_numpy()
tissue_arr = meta["tissue_type"].fill_null("Unknown").to_numpy()
tss_arr = meta["tss"].to_numpy()

print("Numpy arrays gotowe:")
print(f"  stage_arr  ({len(stage_arr)})  unique: {sorted(set(stage_arr))}")
print(f"  event_arr  ({len(event_arr)})  events: {int(event_arr.sum())}, censored: {int((~event_arr).sum())}")
print(f"  gender_arr ({len(gender_arr)}) unique: {sorted(set(gender_arr))}")
print(f"  tissue_arr ({len(tissue_arr)}) unique: {sorted(set(tissue_arr))}")
print(f"  tss_arr    ({len(tss_arr)})    unique TSS: {len(set(tss_arr))}")
print()
print(f"Próbek z brakującym vital_status: {meta['vital_status'].null_count()}")
print(f"Próbek z brakującym stage:        {meta['ajcc_pathologic_stage'].null_count()}")


In [ ]:
# PC1 vs PC2 - 4 panele dla różnych grupowań
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

pc1 = pcs[:, 0]
pc2 = pcs[:, 1]
pc1_lbl = f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)"
pc2_lbl = f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)"

# Panel A: stage
stage_palette = sns.color_palette("Set2", n_colors=len(set(stage_arr)))
for st, color in zip(sorted(set(stage_arr)), stage_palette):
    mask = stage_arr == st
    axes[0, 0].scatter(pc1[mask], pc2[mask], s=20, alpha=0.6, color=color, label=st)
axes[0, 0].set_xlabel(pc1_lbl); axes[0, 0].set_ylabel(pc2_lbl)
axes[0, 0].set_title("PC1 × PC2 po stage")
axes[0, 0].legend(fontsize=8, loc="best", ncol=2)

# Panel B: event (śmierć vs cenzura)
axes[0, 1].scatter(pc1[~event_arr], pc2[~event_arr], s=20, alpha=0.5, color="#7fb069", label="censored (alive)")
axes[0, 1].scatter(pc1[event_arr], pc2[event_arr], s=20, alpha=0.7, color="#b85042", label="event (death)")
axes[0, 1].set_xlabel(pc1_lbl); axes[0, 1].set_ylabel(pc2_lbl)
axes[0, 1].set_title("PC1 × PC2 po event")
axes[0, 1].legend()

# Panel C: gender
for g, color in zip(["female", "male"], ["#c47ab0", "#5a93c4"]):
    mask = gender_arr == g
    axes[1, 0].scatter(pc1[mask], pc2[mask], s=20, alpha=0.6, color=color, label=g)
axes[1, 0].set_xlabel(pc1_lbl); axes[1, 0].set_ylabel(pc2_lbl)
axes[1, 0].set_title("PC1 × PC2 po gender")
axes[1, 0].legend()

# Panel D: tissue_type
for t, color in zip(sorted(set(tissue_arr)), ["#b85042", "#5a8b3c", "#888888"]):
    mask = tissue_arr == t
    axes[1, 1].scatter(pc1[mask], pc2[mask], s=20, alpha=0.6, color=color, label=t)
axes[1, 1].set_xlabel(pc1_lbl); axes[1, 1].set_ylabel(pc2_lbl)
axes[1, 1].set_title("PC1 × PC2 po tissue_type")
axes[1, 1].legend()

plt.tight_layout()
plt.show()


## 6. Batch effect — Tissue Source Site (TSS)

**Najważniejsza analiza całego notebooka dla obrony.**

TCGA-LUAD był zbierany przez ~30 różnych ośrodków. TSS to dwa znaki w pozycjach 5-6 barcode'u TCGA (np. `TCGA-44-1234` → TSS=`44`). Różne ośrodki mogły używać różnych protokołów ekstrakcji RNA, sekwencjonatorów, mogli pobierać próbki z różnych populacji pacjentów.

**Jeśli na PCA widać grupowanie po TSS** — model survival ucząc się na ekspresji może uczyć się **ośrodka, nie biologii**. To jest klasyczny zarzut recenzentów dla TCGA-based modeli.

Co z tym zrobić w notebooku 06: jeśli batch effect widoczny — należy uwzględnić TSS jako kowariant w Cox albo zastosować ComBat/limma do korekty. Decyzja na notebook 06.


In [ ]:
# Liczność per TSS
tss_counts = meta.group_by("tss").len().sort("len", descending=True)
print(f"=== Liczność per TSS (top 15) ===")
print(tss_counts.head(15))
print()
print(f"Łącznie unikalnych TSS: {tss_counts.height}")
print(f"TSS z >=10 próbkami:    {tss_counts.filter(pl.col('len') >= 10).height}")
print(f"TSS z 1-2 próbkami:     {tss_counts.filter(pl.col('len') <= 2).height}")


In [ ]:
# PCA scatter pokolorowane po TSS (top 8 najliczniejszych + 'inne')
fig, ax = plt.subplots(figsize=(11, 8))

top_tss = tss_counts.head(8)["tss"].to_list()

palette = sns.color_palette("tab10", n_colors=len(top_tss))
for tss_code, color in zip(top_tss, palette):
    mask = tss_arr == tss_code
    n = mask.sum()
    ax.scatter(pc1[mask], pc2[mask], s=30, alpha=0.7, color=color,
               label=f"TSS={tss_code} (n={n})")

# Reszta TSS - jako szary
other_mask = ~np.isin(tss_arr, top_tss)
n_other = other_mask.sum()
ax.scatter(pc1[other_mask], pc2[other_mask], s=18, alpha=0.3, color="#888888",
           label=f"pozostałe TSS (n={n_other})")

ax.set_xlabel(pc1_lbl); ax.set_ylabel(pc2_lbl)
ax.set_title("PC1 × PC2 po Tissue Source Site (TSS)\nGrupowanie = batch effect")
ax.legend(loc="best", fontsize=9, ncol=2)
plt.tight_layout()
plt.show()


## 7. Markery LUAD — sanity check biologiczny

Klasyczne markery histologiczne i molekularne LUAD. Jeśli ekspresja tych genów jest sensowna (wysoka tam gdzie być powinna, zmienna gdzie powinna być zmienna) — to dobry znak że dane *są* tym czym mają być.

**Markery:**
- **EGFR** (ENSG00000146648) — receptor wzrostu, częste mutacje LUAD, zwłaszcza u kobiet niepalących
- **KRAS** (ENSG00000133703) — onkogen, najczęstsze mutacje LUAD u palaczy
- **TP53** (ENSG00000141510) — strażnik genomu, mutacje w >50% LUAD
- **ALK** (ENSG00000171094) — fuzje EML4-ALK, ~5% LUAD, celowane terapeutycznie
- **NKX2-1/TTF1** (ENSG00000136352) — czynnik transkrypcyjny tarczycy i pęcherzyków płucnych, marker różnicowania LUAD vs przerzuty
- **SFTPC** (ENSG00000168484) — surfactant protein C, marker pneumocytów typu II (komórki pochodzenia LUAD)


In [ ]:
# Mapping marker -> ENSG (bez sufiksu wersji, np. ENSG00000146648 zamiast .14)
LUAD_MARKERS = {
    "EGFR":     "ENSG00000146648",
    "KRAS":     "ENSG00000133703",
    "TP53":     "ENSG00000141510",
    "ALK":      "ENSG00000171094",
    "ROS1":     "ENSG00000047936",
    "NKX2-1":   "ENSG00000136352",
    "SFTPC":    "ENSG00000168484",
}

# Pliki GDC mają ENSG z wersją (ENSG00000146648.14) - matchujemy po prefiksie
def find_gene_idx(ensg_base, gene_ids):
    for i, gid in enumerate(gene_ids):
        if gid.startswith(ensg_base + ".") or gid == ensg_base:
            return i
    return None

marker_idx = {}
for symbol, ensg in LUAD_MARKERS.items():
    idx = find_gene_idx(ensg, gene_ids)
    if idx is not None:
        marker_idx[symbol] = idx
        print(f"  {symbol:8} -> {gene_ids[idx]:22}  (idx {idx})")
    else:
        print(f"  {symbol:8} -> NIE ZNALEZIONO ({ensg})")


In [ ]:
# Boxplot ekspresji markerów per stage
fig, axes = plt.subplots(2, 4, figsize=(16, 9), sharey=False)
axes = axes.flatten()

stages_order = ["Stage IA", "Stage IB", "Stage IIA", "Stage IIB",
                "Stage IIIA", "Stage IIIB", "Stage IV"]

for i, (symbol, gene_i) in enumerate(marker_idx.items()):
    expr = X_log[gene_i, :]
    df_plot = pl.DataFrame({"expr": expr, "stage": stage_arr}).filter(
        pl.col("stage").is_in(stages_order)
    )

    data_per_stage = [
        df_plot.filter(pl.col("stage") == s)["expr"].to_numpy()
        for s in stages_order
    ]

    axes[i].boxplot(data_per_stage, labels=[s.replace("Stage ", "") for s in stages_order],
                    widths=0.6, patch_artist=True,
                    boxprops=dict(facecolor="#c4a484", alpha=0.7),
                    medianprops=dict(color="#5a3c2a", linewidth=2))
    axes[i].set_title(f"{symbol}", fontsize=11, fontweight="bold")
    axes[i].set_xlabel("Stage")
    axes[i].set_ylabel("log2(TPM+1)")
    axes[i].tick_params(axis="x", rotation=45)

# Ostatni panel (8) - markery wszystkie razem, średnia per próbka (przegląd)
axes[7].axis("off")
axes[7].text(0.5, 0.5,
             f"Markery wyświetlone: {len(marker_idx)}/{len(LUAD_MARKERS)}\n\n"
             "Co interpretować:\n"
             "- NKX2-1, SFTPC: wysokie u większości\n  (markery LUAD)\n"
             "- EGFR, KRAS, TP53: zmienne\n  (różne tumory mają różny profil)\n"
             "- ALK, ROS1: niskie u większości\n  (rzadkie fuzje, nie nadekspresja)",
             ha="center", va="center", fontsize=10, transform=axes[7].transAxes)

plt.suptitle("Markery LUAD - ekspresja per stage", fontsize=13, y=1.00)
plt.tight_layout()
plt.show()


In [ ]:
# Markery per event (alive vs death)
fig, axes = plt.subplots(2, 4, figsize=(16, 8), sharey=False)
axes = axes.flatten()

for i, (symbol, gene_i) in enumerate(marker_idx.items()):
    expr = X_log[gene_i, :]
    data_event = [expr[~event_arr], expr[event_arr]]

    axes[i].boxplot(data_event, labels=["censored", "event"],
                    widths=0.5, patch_artist=True,
                    boxprops=dict(facecolor="#5a8b3c", alpha=0.5),
                    medianprops=dict(color="#2a4a1c", linewidth=2))
    axes[i].set_title(f"{symbol}", fontsize=11, fontweight="bold")
    axes[i].set_ylabel("log2(TPM+1)")

axes[7].axis("off")
plt.suptitle("Markery LUAD - ekspresja per status (event vs censor)", fontsize=13, y=1.00)
plt.tight_layout()
plt.show()


## 8. Heatmapa korelacji top 50 variable genes

Spearman correlation matrix + hierarchical clustering. Klastry koekspresji wskazują na funkcjonalnie powiązane geny (szlaki). Dla LUAD typowo widać:
- Klaster genów proliferacji (cykl komórkowy)
- Klaster genów immunologicznych
- Klaster markerów różnicowania pęcherzykowego


In [ ]:
# Spearman correlation top 50 variable
TOP_FOR_HEATMAP = 50
top50_idx = top_idx[:TOP_FOR_HEATMAP]
X_top50 = X_log[top50_idx, :]  # 50 x n_samples

# Spearman = Pearson na rangach
ranks = np.apply_along_axis(lambda r: np.argsort(np.argsort(r)), 1, X_top50)
corr = np.corrcoef(ranks)

# Skrócone etykiety (gene_id bez wersji)
labels = [gene_ids[i].split(".")[0] for i in top50_idx]

fig, ax = plt.subplots(figsize=(12, 11))
sns.heatmap(corr, cmap="RdBu_r", center=0, vmin=-1, vmax=1, square=True,
            xticklabels=False, yticklabels=labels, cbar_kws={"label": "Spearman ρ"},
            ax=ax)
ax.set_title(f"Korelacja Spearman top {TOP_FOR_HEATMAP} variable genes\n"
             f"(klastry = potencjalne szlaki koekspresji)", fontsize=12)
plt.tight_layout()
plt.show()


## 9. Wnioski i decyzje dla notebooka 06

**Co zostało ustalone w EDA:**

1. Macierz TPM zbudowana, transformacja log2(TPM+1) stabilizuje rozkład
2. Suma TPM per próbka — sprawdzona (sanity check normalizacji)
3. Top 5000 variable genes wybrane — gotowe do feature selection w Cox
4. PCA pokazała główne osie zmienności
5. **TSS batch effect** — kluczowa obserwacja, decyzja przeniesiona do notebooka 06
6. Markery LUAD zachowują się biologicznie sensownie (sanity OK)
7. Heatmapa pokazała klastry koekspresji

**Decyzje do podjęcia w notebooku 06 baseline survival:**

- **Czy uwzględniać TSS jako kowariant Cox** — zależy od tego co zobaczymy w sekcji 6 tego notebooka
- **Próg high/low ekspresji do KM stratification** — mediana / tercyle / optymalny cut-point z walidacją
- **Które geny wziąć na KM curves** — panel ekspercki (EGFR, KRAS, TP53, NKX2-1) plus top 5-10 z univariate Cox p-value
- **Kowarianty kliniczne w Cox** — age + gender + stage (4-poziomowy collapse czy 7-stage?)
- **Stage NOS, Stage Unknown** — osobne kategorie (zgodnie z wcześniejszą decyzją)
- **Walidacja** — k-fold CV (k=5 lub 10), bootstrap dla CI, ewentualnie nested CV dla feature selection
- **Metryka** — concordance index (C-index) + Harrell's C, ewentualnie time-dependent AUC dla 1y, 3y, 5y

**Co NIE wchodzi do notebooka 06 (baseline):** deep learning, multimodalność, WSI. To dopiero po ustanowieniu solidnego baseline Cox.
